# Bloomberg Pull Data

This notebook pulls both:
- macro data
- stock data


Please make sure to install these packages.

*Bloomberg dependencies*
- xbbg>=0.7.0
- blpapi>=3.16.0

In [ ]:
from pathlib import Path
import pandas as pd
from xbbg import blp
from IPython.display import display

ROOT = Path('/run/example') # Please change this to your own path
MACRO_OUTPUT = ROOT / 'raw_macro_data.csv'
STOCK_OUTPUT = ROOT / 'raw_stock_data.csv'

START_DATE = '2000-01-01' 
END_DATE = pd.Timestamp.today().strftime('%Y-%m-%d')
INDEX_TICKERS = ['VNINDEX Index', 'VHINDEX Index', 'VN30 Index']
SECURITY_SUFFIX = ' Equity'

print('Macro output:', MACRO_OUTPUT)
print('Stock output:', STOCK_OUTPUT)
print('Date range   :', START_DATE, '->', END_DATE)

## Macro tickers

These aliases are the final column names that will be saved in `raw_macro_data.csv`.

In [ ]:
MACRO_TICKERS = {
    'Global_Baltic_Dry': 'BDIY Index',
    'USD_CNY_FX': 'USDCNY Curncy',
    'Comm_Brent_Oil': 'CO1 Comdty',
    'US_CPI_YoY': 'CPI YOY Index',
    'Textile_Cotton_Price': 'CT1 Comdty',
    'US_Dollar_Index': 'DXY Curncy',
    'US_FedFunds_Rate': 'FDTR Index',
    'US_GDP_QoQ': 'GDP CQOQ Index',
    'Comm_Copper': 'HG1 Comdty',
    'Hong_Kong_Index': 'HSI Index',
    'Indonesia_Index': 'JCI Index',
    'Comm_Natural_Gas': 'NG1 Comdty',
    'Philippines_Index': 'PCOMP Index',
    'Thailand_Index': 'SET Index',
    'China_Shanghai_Index': 'SHCOMP Index',
    'US_Market_SP500': 'SPX Index',
    'US_Bond_10Y': 'USGG10YR Index',
    'US_RiskFree_3M': 'USGG3M Index',
    'US_Volatility_VIX': 'VIX Index',
    'VN_CPI_YoY': 'VNCPIYOY Index',
    'USD_VND_FX': 'USDVND Curncy',
    'VN_Market_Index': 'VNINDEX Index',
    'VN_MoneySupply_M2': 'VNM2 Index',
    'Comm_Gold_Spot': 'XAU Curncy',
}

pd.Series(MACRO_TICKERS, name='Bloomberg ticker').to_frame()

## Stock field sets


In [ ]:
MARKET_FIELDS = [
    'PX_LAST',
    'PX_OPEN',
    'PX_HIGH',
    'PX_LOW',
    'TOT_RETURN_INDEX_GROSS_DVDS',
    'CUR_MKT_CAP',
    'PX_VOLUME',
    'EQY_SH_OUT',
    'PX_TO_BOOK_RATIO',
    'PE_RATIO',
    'PX_TO_SALES_RATIO',
    'AVERAGE_DIVIDEND_YIELD',
    'AVERAGE_BID_ASK_SPREAD',
    'ENTERPRISE_VALUE',
    'EBITDA',
    'TOT_DEBT_TO_TOT_EQY',
    'VOLATILITY_30D',
    'VOLATILITY_90D',
    'EQY_FREE_FLOAT_PCT',
    'EQY_INIT_PO_DT',
]

CORE_MARKET_FIELDS = [
    'PX_LAST',
    'PX_OPEN',
    'PX_HIGH',
    'PX_LOW',
    'TOT_RETURN_INDEX_GROSS_DVDS',
    'CUR_MKT_CAP',
    'PX_VOLUME',
    'EQY_SH_OUT',
    'PX_TO_BOOK_RATIO',
    'PE_RATIO',
    'PX_TO_SALES_RATIO',
    'AVERAGE_DIVIDEND_YIELD',
    'AVERAGE_BID_ASK_SPREAD',
]

FUNDAMENTAL_FIELDS = [
    'BS_TOT_ASSET',
    'TOT_COMMON_EQY',
    'CASH_AND_MARKETABLE_SECURITIES',
    'BS_TOT_LIAB2',
    'INVENTORIES',
    'BS_ACCTS_REC_EXCL_NOTES_REC',
    'BS_CUR_ASSET_REPORT',
    'BS_CUR_LIAB',
    'BS_NET_FIX_ASSET',
    'BS_INVENTORIES',
    'SALES_REV_TURN',
    'NET_INCOME',
    'COST_OF_GOODS_SOLD',
    'CAPITAL_EXPEND',
    'IS_RD_EXPEND',
    'IS_OPER_INC',
    'GROSS_PROFIT',
    'CF_CASH_FROM_OPER',
    'CF_FREE_CASH_FLOW',
    'RETURN_COM_EQY',
    'RETURN_ON_ASSET',
    'OPER_MARGIN',
    'PROF_MARGIN',
]

STATIC_FIELDS = [
    'INDUSTRY_SECTOR',
    'CUR_EMPLOYEES',
    'SECURITY_NAME',
    'INDUSTRY_GROUP',
    'INDUSTRY_SUBGROUP',
    'GICS_SECTOR_NAME',
    'GICS_INDUSTRY_GROUP_NAME',
    'GICS_SUB_INDUSTRY_NAME',
]

RENAME_MAP = {
    'PX_LAST': 'Price',
    'PX_OPEN': 'Open',
    'PX_HIGH': 'High',
    'PX_LOW': 'Low',
    'TOT_RETURN_INDEX_GROSS_DVDS': 'TRI_Gross',
    'CUR_MKT_CAP': 'Market_Cap',
    'PX_VOLUME': 'Volume',
    'EQY_SH_OUT': 'Shares_Out',
    'PX_TO_BOOK_RATIO': 'P/B',
    'PE_RATIO': 'P/E',
    'PX_TO_SALES_RATIO': 'P/S',
    'AVERAGE_DIVIDEND_YIELD': 'Div_Yield',
    'AVERAGE_BID_ASK_SPREAD': 'Bid_Ask',
    'ENTERPRISE_VALUE': 'EV',
    'EBITDA': 'EBITDA',
    'TOT_DEBT_TO_TOT_EQY': 'D/E_Ratio',
    'VOLATILITY_30D': 'Vol_30D',
    'VOLATILITY_90D': 'Vol_90D',
    'EQY_FREE_FLOAT_PCT': 'Free_Float_Pct',
    'EQY_INIT_PO_DT': 'IPO_Date',
    'BS_TOT_ASSET': 'Assets',
    'TOT_COMMON_EQY': 'Equity',
    'CASH_AND_MARKETABLE_SECURITIES': 'Cash',
    'BS_TOT_LIAB2': 'Debt',
    'INVENTORIES': 'Inventory_Flow',
    'BS_ACCTS_REC_EXCL_NOTES_REC': 'Receivables',
    'BS_CUR_ASSET_REPORT': 'Cur_Assets',
    'BS_CUR_LIAB': 'Cur_Liab',
    'BS_NET_FIX_ASSET': 'PPE',
    'BS_INVENTORIES': 'Inventory_BS',
    'SALES_REV_TURN': 'Sales',
    'NET_INCOME': 'Net_Income',
    'COST_OF_GOODS_SOLD': 'COGS',
    'CAPITAL_EXPEND': 'Capex',
    'IS_RD_EXPEND': 'R&D',
    'IS_OPER_INC': 'Oper_Inc',
    'GROSS_PROFIT': 'Gross_Profit',
    'CF_CASH_FROM_OPER': 'Oper_CF',
    'CF_FREE_CASH_FLOW': 'FCF',
    'RETURN_COM_EQY': 'ROE_Reported',
    'RETURN_ON_ASSET': 'ROA_Reported',
    'OPER_MARGIN': 'Oper_Margin',
    'PROF_MARGIN': 'Net_Margin',
    'INDUSTRY_SECTOR': 'Sector',
    'CUR_EMPLOYEES': 'Employees',
    'SECURITY_NAME': 'Name',
    'INDUSTRY_GROUP': 'Industry_Group',
    'INDUSTRY_SUBGROUP': 'Industry_Subgroup',
    'GICS_SECTOR_NAME': 'GICS_Sector',
    'GICS_INDUSTRY_GROUP_NAME': 'GICS_Industry_Group',
    'GICS_SUB_INDUSTRY_NAME': 'GICS_Sub_Industry',
}

print('Market fields      :', len(MARKET_FIELDS))
print('Fundamental fields :', len(FUNDAMENTAL_FIELDS))
print('Static fields      :', len(STATIC_FIELDS))

## Helper functions

These helpers keep the rest of the notebook readable.

In [ ]:
def _flatten_bdh(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        out = df.copy()
        out.columns = [col[-1] for col in out.columns]
        return out
    return df


def fetch_macro(start_date: str, end_date: str) -> pd.DataFrame:
    frames = []
    for alias, ticker in MACRO_TICKERS.items():
        print(f'Pulling macro: {alias} <- {ticker}')
        df = blp.bdh(tickers=ticker, flds='PX_LAST', start_date=start_date, end_date=end_date)
        if df is None or df.empty:
            continue
        df = _flatten_bdh(df).reset_index().rename(columns={'index': 'Date', 'PX_LAST': alias})
        frames.append(df[['Date', alias]])
    if not frames:
        return pd.DataFrame(columns=['Date'])
    out = frames[0].copy()
    for df in frames[1:]:
        out = out.merge(df, on='Date', how='outer')
    return out.sort_values('Date').reset_index(drop=True)


def _normalize_ticker(x: str) -> str:
    x = str(x).strip()
    if x.endswith('Equity'):
        return x
    return f'{x}{SECURITY_SUFFIX}'


def get_index_universe() -> list[str]:
    tickers = []
    for idx in INDEX_TICKERS:
        print(f'Loading index members: {idx}')
        df = blp.bds(idx, 'INDX_MEMBERS')
        if df is None or df.empty:
            continue
        col = df.columns[0]
        tickers.extend(df[col].dropna().astype(str).tolist())
    tickers = sorted({_normalize_ticker(t) for t in tickers})
    return tickers


def load_static_fields(ticker: str) -> pd.DataFrame:
    rows = []
    for fld in STATIC_FIELDS:
        try:
            val = blp.bdp(ticker, fld)
            if val is None or val.empty:
                continue
            value = val.iloc[0, 0]
        except Exception:
            value = None
        rows.append({'Ticker': ticker, fld: value})
    if not rows:
        return pd.DataFrame({'Ticker': [ticker]})
    out = pd.DataFrame({'Ticker': [ticker]})
    for row in rows:
        for k, v in row.items():
            if k != 'Ticker':
                out[k] = v
    return out


def attempt_ticker_pull(ticker: str, market_fields: list[str]) -> pd.DataFrame:
    market = blp.bdh(tickers=ticker, flds=market_fields, start_date=START_DATE, end_date=END_DATE)
    market = _flatten_bdh(market).reset_index().rename(columns={'index': 'Date'})

    funda = blp.bdh(tickers=ticker, flds=FUNDAMENTAL_FIELDS, start_date=START_DATE, end_date=END_DATE)
    funda = _flatten_bdh(funda).reset_index().rename(columns={'index': 'Date'})

    merged = market.merge(funda, on='Date', how='outer').sort_values('Date').ffill()
    merged['Ticker'] = ticker
    return merged[merged['Date'] >= pd.Timestamp(START_DATE)].reset_index(drop=True)

## Pull macro data

In [ ]:
macro = fetch_macro(START_DATE, END_DATE)
macro.to_csv(MACRO_OUTPUT, index=False)
print('Saved macro rows:', len(macro))
display(macro.head())

## Build stock universe

In [ ]:
universe = get_index_universe()
print('Total tickers:', len(universe))
display(pd.DataFrame({'Ticker': universe[:20]}))

## Pull stock data

This cell pulls market fields, fundamentals, and static fields for every ticker.
If the full market-field pull fails for a ticker, it retries with the smaller proven core market-field set.

In [ ]:
frames = []
failed = []

for i, ticker in enumerate(universe, start=1):
    print(f'[{i}/{len(universe)}] {ticker}')
    try:
        try:
            daily = attempt_ticker_pull(ticker, MARKET_FIELDS)
        except Exception:
            print('  full market block failed, retrying core market fields')
            daily = attempt_ticker_pull(ticker, CORE_MARKET_FIELDS)
        static = load_static_fields(ticker)
        daily = daily.merge(static, on='Ticker', how='left')
        frames.append(daily)
    except Exception as exc:
        print('  failed:', exc)
        failed.append({'Ticker': ticker, 'error': str(exc)})

stock = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print('Pulled stock rows:', len(stock))
print('Failed tickers   :', len(failed))

## Canonical rename and save

In [ ]:
if not stock.empty:
    stock = stock.rename(columns=RENAME_MAP)
    keep_first = ['Date', 'Ticker']
    other_cols = [c for c in stock.columns if c not in keep_first]
    stock = stock[keep_first + other_cols].sort_values(['Ticker', 'Date']).reset_index(drop=True)

stock.to_csv(STOCK_OUTPUT, index=False)
print('Saved stock rows:', len(stock))
print('Saved stock cols:', len(stock.columns))
STOCK_OUTPUT

## Quick inspection

In [ ]:
display(stock.head(20))
print('Stock columns:')
print(list(stock.columns))

if failed:
    failed_df = pd.DataFrame(failed)
    display(failed_df.head(20))